In [13]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.utils import resample
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### Two-way Fixed Effects ###
def IDT_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float), pd.get_dummies(df['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, fe2, ar1):
    params = []
    params.append(ols / (1 + ols))
    params.append(1 / (1 + ols))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
    return params

def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    ind_regs = OLS(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ind_regs_fe2 = IDT_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ind_regs_fe2, ar_1]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ind_regs_fe2[3].params[1], ar_1.roots)
    return regs, parameters

regs_2016, parameters_2016 = compute_regs('2016-12-31', mean_spf_trim, ind_spf_trim)
%store regs_2016
%store parameters_2016

regs_2019, parameters_2019 = compute_regs('2019-12-31', mean_spf_trim, ind_spf_trim)
%store regs_2019
%store parameters_2019

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
%store regs_2022
%store parameters_2022

def new_2022(df):
    a = df[(df['DATE'] >= '2021-09-30') & (df['DATE'] <= '2022-09-30')]
    return df[~df.index.isin(a.index)]

ind_spf_trim22 = new_2022(ind_spf_trim)
mean_spf_trim22 = new_2022(mean_spf_trim)

regs_2022n, parameters_2022n = compute_regs('2022-12-31', mean_spf_trim22, ind_spf_trim22)
%store regs_2022n
%store parameters_2022n

mean_spf_trim22.to_csv('cleaned_data/mean_spf_trim22.csv', sep=',', index=True)
ind_spf_trim22.to_csv('cleaned_data/ind_spf_trim22.csv', sep=',', index=True)

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2016' (list)
Stored 'parameters_2016' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2019' (list)
Stored 'parameters_2019' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_8349/1570018293.py:78: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))


Stored 'regs_2022' (list)
Stored 'parameters_2022' (list)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Stored 'regs_2022n' (list)
Stored 'parameters_2022n' (list)


In [16]:
###############################################
                ## BOOTSTRAP ##
###############################################
def boot(df, R, date):
    RR = R + 1
    boot_regs0 = []
    boot_regs1 = []
    boot_regs2 = []
    boot_regs3 = []
    for i in range(1, RR):
        boot = resample(df['ID'], replace=True, n_samples=len(df), random_state=i)
        boot_df = df[df['ID'].isin(boot)]
        boot_ols = OLS(boot_df, date)
        boot_regs0.append(boot_ols[0].params[1])
        boot_regs1.append(boot_ols[1].params[1])
        boot_regs2.append(boot_ols[2].params[1])
        boot_regs3.append(boot_ols[3].params[1])
    boot_regs = [boot_regs0, boot_regs1, boot_regs2, boot_regs3]
    return boot_regs

boot_regs_2016 = boot(ind_spf_trim, 10000, '2016-12-31')
%store boot_regs_2016

boot_regs_2019 = boot(ind_spf_trim, 10000, '2019-12-31')
%store boot_regs_2019

boot_regs_2022 = boot(ind_spf_trim, 10000, '2022-12-31')
%store boot_regs_2022

boot_regs_2022n = boot(ind_spf_trim22, 10000, '2022-12-31')
%store boot_regs_2022n

Stored 'boot_regs_2016' (list)
Stored 'boot_regs_2019' (list)
Stored 'boot_regs_2022' (list)
Stored 'boot_regs_2022n' (list)


In [17]:
%store -r boot_regs_2016
%store -r boot_regs_2019
%store -r boot_regs_2022
%store -r boot_regs_2022n
print(np.quantile(boot_regs_2016[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2016[3], 0.5))
print(np.quantile(boot_regs_2019[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2019[3], 0.5))
print(np.quantile(boot_regs_2022[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022[3], 0.5))
print(np.quantile(boot_regs_2022n[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022n[3], 0.5))

[-0.23416372 -0.23237005]
-0.23237004941063144
[-0.2421178 -0.2402939]
-0.24033551513079332
[-0.03399195 -0.03236205]
-0.03236204860627404
[-0.18883744 -0.18716935]
-0.18716934780740518


In [23]:
regs_2022n[1][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     15.35
Date:                Tue, 23 Apr 2024   Prob (F-statistic):           9.07e-05
Time:                        10:05:29   Log-Likelihood:                 11385.
No. Observations:                3997   AIC:                        -2.277e+04
Df Residuals:                    3995   BIC:                        -2.275e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0011      0.000     -3.425      0.001      -0.002      -0.000
r_t3          -0.1872      0.048     -3.918      0.000      -0.281      -0.094
==============================================================================
Omnibus:                      534.725   Durbin-Watson:                   0.359
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1822.625
Skew:                           0.663   Prob(JB):                         0.00
Kurtosis:                       6.031   Cond. No.                         133.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [22]:
regs_2016[1][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     32.19
Date:                Tue, 23 Apr 2024   Prob (F-statistic):           1.51e-08
Time:                        10:05:26   Log-Likelihood:                 10173.
No. Observations:                3424   AIC:                        -2.034e+04
Df Residuals:                    3422   BIC:                        -2.033e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0023      0.000     -7.907      0.000      -0.003      -0.002
r_t3          -0.2324      0.041     -5.674      0.000      -0.313      -0.152
==============================================================================
Omnibus:                      124.585   Durbin-Watson:                   0.455
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              257.400
Skew:                          -0.243   Prob(JB):                     1.28e-56
Kurtosis:                       4.252   Cond. No.                         132.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""